# Hamiltonians with PennyLane

Construct Hamiltonians using `qml.Hamiltonian` and compute expectation values on quantum circuits with `default.qubit`.

In [ ]:
import numpy as np
import pennylane as qml

## H2 Hamiltonian (2-qubit representation)

In [ ]:
N_QUBITS = 2
dev = qml.device("default.qubit", wires=N_QUBITS)

H2_HAMILTONIAN = qml.Hamiltonian(
    [-0.81261, 0.17120, -0.22279, 0.17120, 0.04532],
    [
        qml.Identity(0),
        qml.Z(0),
        qml.Z(1),
        qml.Z(0) @ qml.Z(1),
        qml.X(0) @ qml.X(1),
    ],
)

print("Pauli decomposition:")
for coeff, op in zip(H2_HAMILTONIAN.coeffs, H2_HAMILTONIAN.ops):
    print(f"  {float(coeff):+.5f} · {op.name if hasattr(op, 'name') else op}")

## Exact eigenvalues

In [ ]:
H_mat = qml.matrix(H2_HAMILTONIAN, wire_order=[0, 1])
eigenvalues = np.linalg.eigvalsh(H_mat)
print(f"Exact eigenvalues: {np.round(eigenvalues, 6)}")
print(f"Ground state energy: {eigenvalues[0]:.6f}")

## Expectation values on example states

In [ ]:
@qml.qnode(dev)
def expval_circuit(state_prep):
    state_prep()
    return qml.expval(H2_HAMILTONIAN)

e00 = qml.QNode(lambda: qml.expval(H2_HAMILTONIAN), dev)()
print(f"<00|H|00> = {e00:.6f}")

def prep_plusplus():
    qml.Hadamard(wires=0)
    qml.Hadamard(wires=1)

print(f"<++|H|++> = {expval_circuit(prep_plusplus):.6f}")

def prep_11():
    qml.X(wires=0)
    qml.X(wires=1)

print(f"<11|H|11> = {expval_circuit(prep_11):.6f}")

## ZZ + transverse field lattice Hamiltonian

In [ ]:
J, H_FIELD = 1.0, 0.5
LATTICE_HAMILTONIAN = qml.Hamiltonian(
    [J, H_FIELD, H_FIELD],
    [qml.Z(0) @ qml.Z(1), qml.X(0), qml.X(1)],
)

H_lattice_mat = qml.matrix(LATTICE_HAMILTONIAN, wire_order=[0, 1])
evals = np.linalg.eigvalsh(H_lattice_mat)
print(f"H = {J:.1f}·Z₀Z₁ + {H_FIELD:.1f}·(X₀ + X₁)")
print(f"Eigenvalues: {np.round(evals, 6)}")
print(f"Ground state energy: {evals[0]:.6f}")